# 第 12 讲 Optional Lab: Regularization and Validation

这个 notebook 是可复现实验附录。`optional` 表示教师可选择是否把编程复现实验作为课程要求，不表示内容可以省略。Notebook 会从原始 `sklearn` 数据开始，重现 Packet B-D 的主要证据。

教学边界：本材料只用于机器学习课程讨论，不构成临床诊断、治疗、部署或患者护理建议。


## 0. Environment and imports

运行顺序：从上到下。所有数据来自 `sklearn.datasets.load_breast_cancer`，不需要联网。


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, accuracy_score, balanced_accuracy_score, recall_score,
    confusion_matrix, brier_score_loss, log_loss
)
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', message='Inconsistent values')
RANDOM_STATE = 42
asset_dir = Path('../assets/pbl-regularization-validation')
asset_dir


## 1. Load and inspect the dataset

Target coding used here: malignant cancer = 1, benign = 0. This makes malignant sensitivity easy to read.


In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data.copy()
y = (data.target == 0).astype(int)  # malignant = 1, benign = 0

dataset_summary = pd.DataFrame([{
    'dataset': 'Breast Cancer Wisconsin Diagnostic',
    'samples': X.shape[0],
    'features': X.shape[1],
    'malignant_count': int(y.sum()),
    'benign_count': int((1-y).sum()),
    'malignant_rate': float(y.mean()),
    'missing_values': int(X.isna().sum().sum()),
}])
dataset_summary


## 2. Split design

We use a stratified 60/20/20 train/validation/locked-test split. CV and model selection use the development set (train + validation); the locked test is only for the final report.


In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE, stratify=y_temp
)
pd.DataFrame({
    'split': ['train', 'validation', 'locked_test'],
    'n': [len(y_train), len(y_val), len(y_test)],
    'malignant_rate': [y_train.mean(), y_val.mean(), y_test.mean()],
})


## 3. Modeling helpers

L1, L2, and Elastic Net are all logistic regression pipelines with standardization inside the pipeline.


In [ ]:
def make_pipeline(kind, C=1.0, l1_ratio=0.5):
    if kind == 'unregularized':
        clf = LogisticRegression(penalty='l2', C=1e6, solver='lbfgs', max_iter=5000)
    elif kind == 'l1':
        clf = LogisticRegression(penalty='l1', C=C, solver='saga', max_iter=10000, random_state=RANDOM_STATE)
    elif kind == 'l2':
        clf = LogisticRegression(penalty='l2', C=C, solver='lbfgs', max_iter=5000)
    elif kind == 'elasticnet':
        clf = LogisticRegression(penalty='elasticnet', C=C, l1_ratio=l1_ratio, solver='saga', max_iter=10000, random_state=RANDOM_STATE)
    else:
        raise ValueError(kind)
    return Pipeline([('scaler', StandardScaler()), ('clf', clf)])

def evaluate_model(name, pipe, X_eval, y_eval):
    prob = pipe.predict_proba(X_eval)[:, 1]
    pred = (prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_eval, pred, labels=[0, 1]).ravel()
    coef = pipe.named_steps['clf'].coef_[0]
    return {
        'model': name,
        'n': int(len(y_eval)),
        'nonzero_coefficients': int(np.sum(np.abs(coef) > 1e-6)),
        'mean_abs_coefficient': float(np.mean(np.abs(coef))),
        'auc': float(roc_auc_score(y_eval, prob)),
        'accuracy': float(accuracy_score(y_eval, pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_eval, pred)),
        'sensitivity_malignant': float(recall_score(y_eval, pred, pos_label=1)),
        'specificity_benign': float(tn/(tn+fp)),
        'brier': float(brier_score_loss(y_eval, prob)),
        'log_loss': float(log_loss(y_eval, prob)),
        'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn),
    }


## 4. Packet B reproduction: L1/L2/Elastic Net on locked test

Main tension: L1 gives a sparse model, but sparse does not automatically mean more trustworthy for a medical report.


In [ ]:
model_specs = [
    ('Nearly unregularized', 'unregularized', 1e6),
    ('L1 C=1', 'l1', 1.0),
    ('L2 C=1', 'l2', 1.0),
    ('Elastic Net C=1', 'elasticnet', 1.0),
]
fitted = {}
rows = []
for name, kind, C in model_specs:
    pipe = make_pipeline(kind, C=C)
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    rows.append(evaluate_model(name, pipe, X_test, y_test))

packet_b = pd.DataFrame(rows)
packet_b[['model','nonzero_coefficients','auc','sensitivity_malignant','specificity_benign','brier','log_loss']]


In [ ]:
coef_rows = []
for name, pipe in fitted.items():
    coef = pipe.named_steps['clf'].coef_[0]
    for feature, value in zip(X.columns, coef):
        coef_rows.append({
            'model': name,
            'feature': feature,
            'coefficient_standardized_scale': float(value),
            'abs_coefficient': float(abs(value)),
            'is_nonzero': bool(abs(value) > 1e-6),
        })
coef_df = pd.DataFrame(coef_rows)
coef_df[coef_df['model'].isin(['L1 C=1','L2 C=1'])].sort_values(['model','abs_coefficient'], ascending=[True, False]).groupby('model').head(8)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(packet_b))
ax.bar(x - 0.25, packet_b['sensitivity_malignant'], width=0.25, label='Sensitivity')
ax.bar(x, packet_b['specificity_benign'], width=0.25, label='Specificity')
ax.bar(x + 0.25, packet_b['brier'], width=0.25, label='Brier lower')
ax.set_xticks(x)
ax.set_xticklabels(packet_b['model'], rotation=25, ha='right')
ax.set_ylim(0, 1.05)
ax.set_title('Packet B: locked test metrics')
ax.legend()
fig.tight_layout()


## 5. Packet C reproduction: CV path and nested CV

C is inverse regularization strength: larger C means weaker penalty. Report mean and uncertainty, not only the best mean.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
C_values = np.logspace(-3, 2, 13)
path_rows = []
for penalty, kind in [('L1', 'l1'), ('L2', 'l2'), ('Elastic Net', 'elasticnet')]:
    for C in C_values:
        pipe = make_pipeline(kind, C=float(C))
        scores = cross_validate(pipe, X_temp, y_temp, scoring=['roc_auc','balanced_accuracy'], cv=cv, return_estimator=True)
        nonzero = [int(np.sum(np.abs(est.named_steps['clf'].coef_[0]) > 1e-6)) for est in scores['estimator']]
        path_rows.append({
            'penalty': penalty,
            'C': float(C),
            'cv_auc_mean': float(np.mean(scores['test_roc_auc'])),
            'cv_auc_sd': float(np.std(scores['test_roc_auc'], ddof=1)),
            'cv_balanced_accuracy_mean': float(np.mean(scores['test_balanced_accuracy'])),
            'cv_balanced_accuracy_sd': float(np.std(scores['test_balanced_accuracy'], ddof=1)),
            'nonzero_coefficients_mean': float(np.mean(nonzero)),
            'nonzero_coefficients_sd': float(np.std(nonzero, ddof=1)),
        })
cv_path = pd.DataFrame(path_rows)
cv_path.sort_values('cv_auc_mean', ascending=False).head(10)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for penalty, g in cv_path.groupby('penalty'):
    axes[0].errorbar(g['C'], g['cv_auc_mean'], yerr=g['cv_auc_sd'], marker='o', capsize=2, label=penalty)
    axes[1].plot(g['C'], g['nonzero_coefficients_mean'], marker='o', label=penalty)
axes[0].set_xscale('log')
axes[0].set_title('5-fold CV AUC mean +/- SD')
axes[0].set_xlabel('C larger = weaker regularization')
axes[0].set_ylabel('AUC')
axes[1].set_xscale('log')
axes[1].set_title('Nonzero coefficients across C')
axes[1].set_xlabel('C')
for ax in axes: ax.legend()
fig.tight_layout()


In [ ]:
param_grid = [
    {'clf__penalty':['l1'], 'clf__solver':['saga'], 'clf__C':[0.01,0.03,0.1,0.3,1,3,10]},
    {'clf__penalty':['l2'], 'clf__solver':['lbfgs'], 'clf__C':[0.01,0.03,0.1,0.3,1,3,10]},
    {'clf__penalty':['elasticnet'], 'clf__solver':['saga'], 'clf__l1_ratio':[0.5], 'clf__C':[0.01,0.03,0.1,0.3,1,3,10]},
]
base = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=10000, random_state=RANDOM_STATE))])
inner = StratifiedKFold(n_splits=4, shuffle=True, random_state=7)
outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=8)
nested_rows = []
for fold, (tr, te) in enumerate(outer.split(X_temp, y_temp), start=1):
    gs = GridSearchCV(base, param_grid, scoring='roc_auc', cv=inner)
    gs.fit(X_temp.iloc[tr], y_temp.iloc[tr])
    prob = gs.predict_proba(X_temp.iloc[te])[:, 1]
    pred = (prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_temp.iloc[te], pred, labels=[0, 1]).ravel()
    nested_rows.append({
        'outer_fold': fold,
        'best_penalty': gs.best_params_.get('clf__penalty'),
        'best_C': gs.best_params_.get('clf__C'),
        'best_l1_ratio': gs.best_params_.get('clf__l1_ratio', ''),
        'outer_auc': roc_auc_score(y_temp.iloc[te], prob),
        'outer_balanced_accuracy': balanced_accuracy_score(y_temp.iloc[te], pred),
        'outer_sensitivity_malignant': recall_score(y_temp.iloc[te], pred, pos_label=1),
        'outer_specificity_benign': tn/(tn+fp),
    })
nested_cv = pd.DataFrame(nested_rows)
nested_cv


In [ ]:
nested_summary = nested_cv[['outer_auc','outer_balanced_accuracy','outer_sensitivity_malignant','outer_specificity_benign']].agg(['mean','std']).T
nested_summary


## 6. Packet D reproduction: bootstrap CI, calibration, and stress validation

Bootstrap CI is estimated on the locked test set. It quantifies uncertainty; it does not create external validation.


In [ ]:
def bootstrap_ci(pipe, X_test, y_test, model_name, n_boot=2000, seed=20260816):
    rng = np.random.default_rng(seed)
    prob_all = pipe.predict_proba(X_test)[:, 1]
    pred_all = (prob_all >= 0.5).astype(int)
    y_arr = np.asarray(y_test)
    metrics = {'auc': [], 'sensitivity_malignant': [], 'specificity_benign': [], 'brier': [], 'balanced_accuracy': []}
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_arr), len(y_arr))
        yy = y_arr[idx]
        if len(np.unique(yy)) < 2:
            continue
        pp = prob_all[idx]
        pr = pred_all[idx]
        tn, fp, fn, tp = confusion_matrix(yy, pr, labels=[0, 1]).ravel()
        metrics['auc'].append(roc_auc_score(yy, pp))
        metrics['sensitivity_malignant'].append(recall_score(yy, pr, pos_label=1))
        metrics['specificity_benign'].append(tn/(tn+fp))
        metrics['brier'].append(brier_score_loss(yy, pp))
        metrics['balanced_accuracy'].append(balanced_accuracy_score(yy, pr))
    point = evaluate_model(model_name, pipe, X_test, y_test)
    rows = []
    for metric, values in metrics.items():
        arr = np.asarray(values, dtype=float)
        rows.append({
            'model': model_name,
            'metric': metric,
            'estimate': point[metric],
            'ci_low_2.5': np.percentile(arr, 2.5),
            'ci_high_97.5': np.percentile(arr, 97.5),
            'n_bootstrap_valid': len(arr),
        })
    return rows

boot_rows = []
for name in ['L1 C=1', 'L2 C=1', 'Elastic Net C=1']:
    boot_rows.extend(bootstrap_ci(fitted[name], X_test, y_test, name))
bootstrap_results = pd.DataFrame(boot_rows)
bootstrap_results


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_df = bootstrap_results[bootstrap_results['metric'].isin(['auc','sensitivity_malignant','specificity_benign','brier'])].reset_index(drop=True)
for i, row in plot_df.iterrows():
    ax.errorbar(row['estimate'], i, xerr=[[row['estimate']-row['ci_low_2.5']], [row['ci_high_97.5']-row['estimate']]], fmt='o', capsize=3)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df['model'] + ' | ' + plot_df['metric'])
ax.set_xlim(0, 1.05)
ax.set_title('Locked test bootstrap 95% CI')
fig.tight_layout()


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for name in ['L1 C=1', 'L2 C=1']:
    prob = fitted[name].predict_proba(X_test)[:, 1]
    frac, meanp = calibration_curve(y_test, prob, n_bins=5, strategy='quantile')
    ax.plot(meanp, frac, marker='o', label=name)
ax.plot([0,1], [0,1], '--', color='gray', label='ideal')
ax.set_xlabel('Mean predicted malignant probability')
ax.set_ylabel('Observed malignant fraction')
ax.set_title('Calibration check')
ax.legend()
fig.tight_layout()


In [ ]:
radius = X['mean radius']
high_radius = radius >= radius.quantile(0.70)
source_like = ~high_radius
stress_rows = []
for name, kind, C in [('L1 tuned C=0.3','l1',0.3), ('L2 tuned C=0.3','l2',0.3), ('Elastic Net tuned C=0.3','elasticnet',0.3)]:
    pipe = make_pipeline(kind, C=C)
    pipe.fit(X.loc[source_like], y.loc[source_like])
    for split_name, mask in [('source-like internal subset', source_like), ('high-radius stress subset', high_radius)]:
        row = evaluate_model(name, pipe, X.loc[mask], y.loc[mask])
        row['validation_split'] = split_name
        row['malignant_rate'] = float(y.loc[mask].mean())
        stress_rows.append(row)
stress_validation = pd.DataFrame(stress_rows)
stress_validation[['model','validation_split','n','malignant_rate','auc','balanced_accuracy','sensitivity_malignant','specificity_benign','brier']]


## 7. Compare with generated packet assets

The package assets are stored as CSV/PNG. Use this section to inspect the exact tables embedded in the worksheet and evidence pack.


In [ ]:
generated_assets = sorted(asset_dir.glob('*.csv'))
pd.DataFrame({'csv_asset': [p.name for p in generated_assets]})


In [ ]:
pd.read_csv(asset_dir / 'packet-d-locked-test-bootstrap-ci.csv').head(20)


## 8. One-page validation protocol template

Use this as the homework structure. Keep it to one page.

1. Biomedical task and target:
2. Data size, feature count, missingness, class balance:
3. Split/CV plan and how the test set is locked:
4. Regularization candidates and hyperparameter range:
5. Metrics and uncertainty report:
6. Calibration check:
7. External validation or reason it is unavailable:
8. Boundary sentence: what this evidence cannot claim:


## Final reflection

- Which model would you report: L1, L2, Elastic Net, or insufficient evidence?
- Which single metric would be most dangerous to report alone?
- What sentence prevents the report from overclaiming clinical readiness?
